In [0]:
dbutils.widgets.text('p_batch_id', '')
v_batch_id = dbutils.widgets.get('p_batch_id')

In [0]:
%run ../common/environmental_config

In [0]:
%run ../common/00.bronze-helper

In [0]:
#define source_fie and table_name
source_file = f"{loading_folder_path}/{v_batch_id}/results"
table_name = f"{catalog_name}.{bronze_schema}.results"

In [0]:
#schema
from pyspark.sql.types import StructType , StructField , DateType , StringType , IntegerType , FloatType
results_schema = StructType([
    StructField('date' , DateType()),
    StructField('raceName' , StringType()),
    StructField('round' , IntegerType()),
    StructField('season' , IntegerType()),
    StructField('url' , StringType()),
    StructField('constructorId' , StringType()),
    StructField('driverId' , StringType()),
    StructField('grid' , IntegerType()),
    StructField('laps' , IntegerType()),
    StructField('number' , IntegerType()),
    StructField('points' , FloatType()),
    StructField('position' , IntegerType()),
    StructField('positionText' , StringType()),
    StructField('status' , StringType())
])

In [0]:
df_results = (
    spark.read
         .format('json')
         .schema(results_schema)
         .option('mode' , 'failfast')
         .load(source_file)
)

In [0]:
final_df_results = add_ingestion_metadata(df_results)

In [0]:
#write to detla table
write_to_bronze (input_df = final_df_results,
                 table_name = table_name,
                 batch_id = v_batch_id)